# Documents Notebook

For this chatbot I will be using bash man pages, tealdeer output and the output from the bash help system for builtin commands.

There will need to be 2 pipelines.
- The first will be the data ingestion pipeline, which is in this notebook.
- Then we will need a query pipeline that will use our domain specific emnbeddings as the context for an LLM prompt. This is the RAG implementation that will be built. It is located in bashbot.ipynb and the associated script.

The dependencies are in requirements.txt.

### Architecture

![image](../images/architecture.png)

## Data Ingestion Pipeline for Bashbot

### Imports

In [1]:
#pip install langchain-community
#!pip install chromadb

In [2]:
import os
import sys
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, TextLoader #,PyPDFLoader, PyMuPDFLoader for PDFs in the future
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Function Definitions

### load the corpus from the text files in the data directory

In [3]:
def load_txt_corpus():
    """ Loads all text files from the ./data directory."""
    loader = DirectoryLoader(
        "../data",
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={'encoding': 'utf-8'},
        show_progress=True
    )
    corpus = loader.load()

    return corpus

In [4]:
def process_text_files(path):
    """Processes all text (.txt) files in the provided path"""
    documents = []
    file_dir = Path(path)

    files = list(file_dir.glob("**/*.txt"))
    print(f"Preparing to process {len(files)} text files...")

    # process the files and modify the source metadata to include manpage reference if possible.
    for f in files:
#        print(f"processing {f.name}")
        try:
            loader = TextLoader(str(f))
            docs = loader.load()

            for d in docs:
                d.metadata['manpage'] = Path(f).stem
                d.metadata['source_file'] = f.name
                d.metadata['file_type'] = 'txt'
                d.metadata['platform'] = 'linux'
                d.metadata['source'] = 'manpage'

                documents.extend(docs)

        except Exception as e:
            print(f"    ERROR {e}")

    print(f"Total documents loaded: {len(documents)}.")
    print(type(documents))
    return list(documents)
    


In [5]:
def process_all_files():
    documents = []
    documents.extend(process_text_files("../data"))

    return documents

### divide text into chunks

In [6]:
# Splits the douments into manageable sized chunks.
def split_documents(corpus, chunk_size=500, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(corpus)
    print(f"Splitting {len(corpus)} documents into {len(split_docs)} chunks.")
    if split_docs:
        print("\nSample chunk:")
        print(f"Page_content: {split_docs[0].page_content[:300]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

### main()

In [7]:
# Obligatory entry point
def main():
   # load_txt_corpus()
    corpus = process_all_files()

In [8]:
#if __name__ == "__main__":
#   main()

## Notebook Exploration

### NOTE
If you want to use the existing vector db and skip parsing the docs, set the __USE_EXISTING=True__ below

In [9]:
USE_EXISTING=False
VERBOSE = False

**Read data from files, and maybe an RDB**

In [10]:
corpus = process_all_files()

Preparing to process 2077 text files...
Total documents loaded: 2077.
<class 'list'>


**Chunking**

In [11]:
chunks = split_documents(corpus)

Splitting 2077 documents into 39773 chunks.

Sample chunk:
Page_content: request-key - handle key instantiation callback requests from the kernel
/sbin/request-key <op> <key> <uid> <gid> <threadring> <processring>
     <sessionring> [<info>] This program is invoked by the kernel when the
kernel is asked for a key that it doesn't have immediately available. The
kernel cre...
Metadata: {'source': 'manpage', 'manpage': 'request-key', 'source_file': 'request-key.txt', 'file_type': 'txt', 'platform': 'linux'}


In [12]:
chunks[3]

Document(metadata={'source': 'manpage', 'manpage': 'request-key', 'source_file': 'request-key.txt', 'file_type': 'txt', 'platform': 'linux'}, page_content="the current directory.\tThe program will use from the current directory rather\nthan from Don't log to the system log.\tOrdinarily, error messages and\ndebugging messages will be copied to the system log - this will prevent that.\nTurn on debugging output.  This may be specified multiple times to produce\nincreasing levels of verbosity.\t Print the program version and exit.  All\nerrors will be logged to the syslog.  /etc/request-key.d/*.conf Individual")

### Log Classes

### Embedding Manager

I want the flexibility to have pluggable vector store implementations. I am going to wrap this part in classes.
Inspiration for this came from Krish Naik's YouTube video
https://youtu.be/o126p1QN_RI?si=7xC5H47A3iuu52RK
and pixegami https://youtu.be/2TJxpyO3ei4?si=zPNEsAFWWy5Cenzq


In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from chromadb.utils.batch_utils import create_batches
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class EmbeddingMgr:
    """This class manages the embedding implementation.
    For now I am going to use a SentenceTransformer of
    some kind."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", verbose: bool = False):
        """
        ctor

        Args:
            model_name: model name for sentence embeddings.

        Notes: For now I am using HuggingFace models.
        """

        self.model_name = model_name
        self.model = None
        self.verbose = verbose
        self._load_model()
        
        
    def _load_model(self):
        """This method loads the model defined in the ctor."""
        try:
            print(f"Loading model {self.model_name}.")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.")
            print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Exception while loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of text strings

        Args:
            texts: List of strings

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model has not been loaded.")

        if VERBOSE: 
            print(f"Generating embeddings for {len(texts)} strings.")
        embeddings = self.model.encode(texts, show_progress_bar=self.verbose)
        if VERBOSE:
            print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    


Initialize the Embedding Manager

In [15]:
emb_mgr = EmbeddingMgr("all-MiniLM-L6-v2", verbose=False)
emb_mgr

Loading model all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.
Embedding dimension: 384


### VectorSrore Database

In [16]:
class VectorStore:
    """Manages embeddings in a vector database"""

    def __init__(self, collection_name: str = "bashdocs", persist_directory: str = "../data/vector_store"):
        """
        ctor

        Args:
            collection_name: Name of the collection
            persist_directory: Path to directory for persistant vectors.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            # Create the persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Bash commands document embeddings for RAG"}
            )
            print(f"Vector store insitialized for collection {self.collection_name}")
            print(f"Exisiting documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
            
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents to the vector store

        Args:
            documents: List of langchain Documents
            embeddings: The embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents to embeddings mismatch.")

        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for idx, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate the uuid
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{idx}"
            ids.append(doc_id)

            # Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = idx
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document Page Content
            documents_text.append(doc.page_content)

            # Embedding vetor
            embeddings_list.append(embedding.tolist())

        # Add it to the collection
        try:
            max_batch = 5000 # batch max is 5461
            for i in range(0, len(embeddings), max_batch):
                self.collection.add(
                    embeddings=embeddings_list[i:i + max_batch],
                    documents=documents_text[i:i + max_batch],
                    ids=ids[i:i + max_batch],
                    metadatas=metadatas[i:i + max_batch]
                )
            print(f"Successfully added {len(documents)} documents to the store.")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e})")
            raise
            


In [17]:
vector_store = VectorStore()
vector_store

Vector store insitialized for collection bashdocs
Exisiting documents in collection: 0


### Create embeddings

In [18]:
# get the text from the documents
if not USE_EXISTING:
    texts = [doc.page_content for doc in chunks]

    # Use the EmbeddingMgr class to generate them
    embeddings = emb_mgr.generate_embeddings(texts)

    # Add the documents to the vector store
    vector_store.add_documents(chunks, embeddings)

Adding 39773 documents to the vector store...
Successfully added 39773 documents to the store.
Total documents in collection: 39773


## Prompt and RAG Retrieval Pipeline

### Retriever class

In [19]:
class Retriever:
    """Encodes a query and returns similar documents from the vector store db"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingMgr):
        """
        ctor

        Args:
            vector_store: The VectorStore of embeddings
            embedding_manager: The EmbeddingMgr that handles encoding embeddings
        """
        self.store = vector_store
        self.manager = embedding_manager

    def fetch(self, query: str, top_k: int = 5, threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieves documents from the store

        Args:
            query: The user query
            top_k: Number of results to return
            threshold: Minimum similarity score

        Returns:
            List of dictionaries with the documents and metadata
        """
        if VERBOSE:
            print(f"Fetching documents for '{query}'")
            print(f"Top K: {top_k}. Threshold: {threshold}")

        # create the embedding for the query
        query_embedding = self.manager.generate_embeddings([query])[0]

        # Search the vector store
        try:
            results = self.store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            fetched_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score - cosine distance for chromadb
                    similarity = 1 - distance

                    if similarity >= threshold:
                        fetched_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity,
                            'distance': distance,
                            'rank': i + 1
                        })
                if VERBOSE:
                    print(f"Fetched {len(fetched_docs)} documents (filtered by threshold)")
            else:
                if VERBOSE:
                    print("No documents found")
                    
            return fetched_docs

        except Exception as e:
            print(f"Error fetching documents: {e}")
            return []
            

In [20]:
# RAG function to pull it all together
def rag_call(query, retriever, llm, top_k=3, pure: bool = False) -> Tuple:
    """Calls the LLM

    Args:
        query: the question asked
        llm: the llm instance to call
        top_k: the number of documents to return for the context
        pure: set to True to call the llm without context for testing
    """
    
    # RAG function to pull it all together
    if not pure:
        fetched_docs = retriever.fetch(query, top_k=top_k)
    else:
        fetched_docs = []
        
    # put all fetched documents into the context
    if fetched_docs: 
        found = len(fetched_docs)
        context = "\n\n".join(doc['content'] for doc in fetched_docs)
    else:
        found  = 0
        context = ""
        
    # Create the prompt
    prompt = f"""You are a proffesional AI assistant theat specializes in asnwering
        questions about Linux commands.
        Use the following context to answer the question accurately and concisely.
        Context:
        {context}
        
        Question: {query}
        
        Answer:"""
    try:
        response = llm.invoke([prompt.format(context=context, query=query)])
    except Exception as e:
        print(f"Error:{e}\nContext:{context}")
        
    return (found, response.content, context)

## Read the JSON questions and answers

We need to call the function **rag_call(query, retriever, llm, top_k=3)** with each query from the JSON file and capture the LLM answer and the answer from the JSON test answer for each question and compare them. We may be able to use cosine similarity, but a manual inspection will probably be needed for accurate assessment.

### submit a single question and return the answer

In [21]:
import json
from langchain_groq import ChatGroq

In [22]:
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=api_key, model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)
fido = Retriever(vector_store, emb_mgr)

### Load the JSON test data

In [23]:
with open("../test/test_questions.json", "r") as jsf:
    test_data = json.load(jsf)

In [24]:
type(test_data)

dict

In [25]:
import pandas as pd

df_test = pd.DataFrame(test_data)

In [26]:
df_test.head()

,linux_questions
0,"{'id': 1, 'question': 'What is Linux?', 'answe..."
1,"{'id': 2, 'question': 'What is a Linux distrib..."
2,"{'id': 3, 'question': 'What is the difference ..."
3,"{'id': 4, 'question': 'What is the Linux shell..."
4,"{'id': 5, 'question': 'How do you check the cu..."


In [27]:
results = []
for item in test_data['linux_questions'][:20]:
    result = rag_call(item['question'], fido, llm)
    item['result'] = result
    results.append(item)

In [28]:
len(results)

20

In [29]:
df_results = pd.DataFrame(results)
df_results.head()

,id,question,answer,result
0,1,What is Linux?,"Linux is an open-source, Unix-like operating s...","(2, Based on the context provided, it seems th..."
1,2,What is a Linux distribution?,A Linux distribution (distro) bundles the Linu...,"(0, A Linux distribution, often abbreviated as..."
2,3,What is the difference between the kernel and ...,"The kernel manages hardware/resources (CPU, me...","(0, In the context of Linux, the kernel and th..."
3,4,What is the Linux shell?,"The shell is a command interpreter (e.g., bash...","(3, The Linux shell is a command-line interpre..."
4,5,How do you check the current working directory?,Use 'pwd'.,"(3, To check the current working directory, yo..."


## Function Definitions

#### Helper to calculate cosine similarity between the provided JSON answer and the LLM answer

In [30]:
def calculate_similarity(answer, llm):
    """Calculates the cosine similarity between 2 embeddings"""
    dot_product = np.dot(answer, llm)
    norm_answer = np.linalg.norm(answer)
    norm_llm = np.linalg.norm(llm)
    if norm_answer == 0 or norm_llm == 0:
        return 0
    return dot_product / (norm_answer * norm_llm)


#### Test Loop

In [31]:
def test_similarity(data: Dict = None, iterations: int = 20, pure: bool = False, top_k: int = 3) -> List[Dict]:
    results = []
    manager = EmbeddingMgr(model_name="all-MiniLM-L6-v2", verbose=False)
    store = VectorStore()
    api_key = os.getenv("GROQ_API_KEY")
    llm = ChatGroq(groq_api_key=api_key, model_name="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)
    fido = Retriever(store, manager)

    for item in data['linux_questions'][:iterations]:
        result = rag_call(item['question'], retriever=fido, top_k=top_k, llm=llm, pure=pure)
        X = manager.generate_embeddings(item['answer'])
        Y = manager.generate_embeddings(result[1])
        similarity = calculate_similarity(X, Y)
        item['result'] = result
        item['similarity'] = similarity
        item['context'] = result[2]
        results.append(item)

    
    return results

In [32]:
answer_emb = emb_mgr.generate_embeddings(df_results['answer'][0])
llm_emb = emb_mgr.generate_embeddings(df_results['result'][0][1])

In [33]:
similarity = calculate_similarity(answer_emb, llm_emb)

In [34]:
print(similarity)

0.7493443


In [35]:
results = test_similarity(test_data, iterations=20, pure=False)

Loading model all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.
Embedding dimension: 384
Vector store insitialized for collection bashdocs
Exisiting documents in collection: 39773


In [36]:
len(results)

20

#### Average Similrity (with RAG)

In [37]:
df = pd.DataFrame(results)
df.head()
print(df['similarity'].mean())

0.6800529


In [38]:
df['similarity'].describe()

count    20.000000
mean      0.680053
std       0.172849
min       0.359682
25%       0.523222
50%       0.725143
75%       0.812231
max       0.939828
Name: similarity, dtype: float64

In [39]:
df['similarity'].value_counts()

similarity
0.939828    1
0.820772    1
0.809384    1
0.780814    1
0.520586    1
0.435932    1
0.631838    1
0.716226    1
0.785968    1
0.822486    1
0.508566    1
0.359682    1
0.734061    1
0.442210    1
0.791427    1
0.896538    1
0.884839    1
0.635151    1
0.524101    1
0.560650    1
Name: count, dtype: int64

### Test without RAG

In [40]:
results = test_similarity(test_data, iterations=20, pure=False)

Loading model all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.
Embedding dimension: 384
Vector store insitialized for collection bashdocs
Exisiting documents in collection: 39773


#### Average Similrity (without RAG)

In [41]:
df = pd.DataFrame(results)
df.head()
print(df['similarity'].mean())

0.6763648


In [42]:
df['similarity'].describe()

count    20.000000
mean      0.676365
std       0.161610
min       0.396538
25%       0.522490
50%       0.714589
75%       0.797428
max       0.905146
Name: similarity, dtype: float64

In [43]:
df['similarity'].value_counts()

similarity
0.794656    1
0.825107    1
0.853091    1
0.775035    1
0.516152    1
0.436016    1
0.685717    1
0.716226    1
0.782314    1
0.805741    1
0.517659    1
0.443005    1
0.712951    1
0.396538    1
0.778736    1
0.905146    1
0.884839    1
0.613616    1
0.524101    1
0.560650    1
Name: count, dtype: int64

### Test with RAG and varying top_k

In [ ]:
results = test_similarity(test_data, top_k=2, iterations=20, pure=False)

In [ ]:
df = pd.DataFrame(results)
df.head()
print(df['similarity'].mean())

In [ ]:
df['similarity'].describe()

In [ ]:
df['similarity'].value_counts()

In [44]:
results = test_similarity(test_data, iterations=len(test_data['linux_questions']), pure=False)

Loading model all-MiniLM-L6-v2.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.
Embedding dimension: 384
Vector store insitialized for collection bashdocs
Exisiting documents in collection: 39773


In [45]:
len(results)

50

### Citations

IBM Technology
LangChain vs LangGraph: A Tale of Two Frameworks
https://youtu.be/qAF1NjEVHhY?si=CKShLFYuOz0xbi1N

Krish Naik
Complete RAG Crash Course With Langchain In 2 Hours
https://youtu.be/o126p1QN_RI?si=7xC5H47A3iuu52RK

pixegami
Python RAG Tutorial (with Local LLMs): AI For Your PDFs
https://youtu.be/2TJxpyO3ei4?si=zPNEsAFWWy5Cenzq


Technologies, Cuantum. Natural Language Processing with Python Updated Edition: From Basics to Advanced Projects: Mastering Text Analysis, Machine Learning Models, and Chatbot ... (Mastering the AI Revolution Book 4) (p. 1). (Function). Kindle Edition. 


Tunstall, Lewis; Werra, Leandro von; Wolf, Thomas. Natural Language Processing with Transformers, Revised Edition (p. 4). (Function). Kindle Edition. 